In [10]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder, StandardScaler

df = pd.read_csv('wbc.csv')
df.info()
df.head()

<class 'pandas.DataFrame'>
RangeIndex: 569 entries, 0 to 568
Data columns (total 33 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   id                       569 non-null    int64  
 1   diagnosis                569 non-null    str    
 2   radius_mean              569 non-null    float64
 3   texture_mean             569 non-null    float64
 4   perimeter_mean           569 non-null    float64
 5   area_mean                569 non-null    float64
 6   smoothness_mean          569 non-null    float64
 7   compactness_mean         569 non-null    float64
 8   concavity_mean           569 non-null    float64
 9   concave points_mean      569 non-null    float64
 10  symmetry_mean            569 non-null    float64
 11  fractal_dimension_mean   569 non-null    float64
 12  radius_se                569 non-null    float64
 13  texture_se               569 non-null    float64
 14  perimeter_se             569 non-null

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,NaN
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,NaN
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,NaN


##### Differentiate between variables that are usable and variables that are not usable.

In [11]:
# Variabel yang tidak usable
unsable = df[['id', 'Unnamed: 32']]
df = df.drop(columns=unsable)

##### Perform encoding on the "diagnosis" column.

In [12]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df['diagnosis'] = le.fit_transform(df['diagnosis'])

print(df['diagnosis'].unique())

[1 0]


##### Perform standardization on all columns that possess numerical values.

In [13]:
from sklearn.preprocessing import StandardScaler

features = df.drop(columns=['diagnosis'])

scaler = StandardScaler()
features_scaled = scaler.fit_transform(features)

X = pd.DataFrame(features_scaled, columns=features.columns)
y = df['diagnosis']

##### Perform the feature selection process. You may use SelectKBest.

In [14]:
from sklearn.feature_selection import SelectKBest, f_classif

In [15]:
selector = SelectKBest(score_func=f_classif, k=10)

X_selected = selector.fit_transform(X, y)

selected_features = X.columns[selector.get_support()]

print("Fitur terpilih:")
print(selected_features)

Fitur terpilih:
Index(['radius_mean', 'perimeter_mean', 'area_mean', 'concavity_mean',
       'concave points_mean', 'radius_worst', 'perimeter_worst', 'area_worst',
       'concavity_worst', 'concave points_worst'],
      dtype='str')


##### Conduct the testing process using a Logistic Regression model as in practical session 1.

In [16]:
import pandas as pd
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression

# Load data
df = pd.read_csv('wbc.csv')

# Hapus kolom yang tidak digunakan
df = df.drop(columns=['id', 'Unnamed: 32'])

# Encoding diagnosis
df['diagnosis'] = df['diagnosis'].map({
    'B': 0,
    'M': 1
})

# Pisahkan fitur dan target
X = df.drop(columns=['diagnosis'])
y = df['diagnosis']

# Cross validation
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

results = []

for k in range(1, X.shape[1] + 1):

    pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('selector', SelectKBest(f_classif, k=k)),
        ('model', LogisticRegression(max_iter=5000))
    ])

    scores = cross_val_score(
        pipeline,
        X,
        y,
        cv=cv,
        scoring='accuracy'
    )

    results.append({
        'k': k,
        'accuracy': scores.mean()
    })

results_df = pd.DataFrame(results)

print(results_df)

     k  accuracy
0    1  0.906878
1    2  0.931517
2    3  0.931501
3    4  0.940273
4    5  0.942028
5    6  0.943782
6    7  0.947291
7    8  0.949045
8    9  0.954308
9   10  0.950815
10  11  0.950800
11  12  0.954324
12  13  0.949061
13  14  0.947306
14  15  0.947306
15  16  0.957833
16  17  0.966636
17  18  0.970144
18  19  0.971899
19  20  0.975423
20  21  0.971914
21  22  0.971914
22  23  0.975423
23  24  0.977177
24  25  0.978932
25  26  0.980686
26  27  0.978932
27  28  0.975423
28  29  0.975423
29  30  0.973669


##### You may employ a model pipeline to facilitate your work.
saat melakukan pengujian pendekatan SelectBest + StandartScaler + Logistic Regression + 5-fold cross_validation pada dataset, Hasil terbaik K = 26 fitur, dengan rata accuracy sekitar 98,07%

| Jumlah fitur (K) |   Accuracy |
| ---------------: | ---------: |
|               24 |     97,72% |
|               25 |     97,89% |
|           **26** | **98,07%** |
|               27 |     97,89% |
|               28 |     97,54% |
|               29 |     97,54% |
|               30 |     97,37% |


##### Based on your result, how many optimal feature can be used?, which feature are they?

Pada dataset WBC, variabel id dan Unnamed: 32 tidak digunakan karena id hanya identitas data dan Unnamed: 32 tidak memiliki informasi, Var diagnosis digunakan sebagai target dan menjadi nilai numerik, yaitu *B = 0*, dan *M = 1*, seluruh 30 var pengukuran numerik distandardisasi

Lalu pemilihan fitur menggunakan `SelectKBest` dan `f_classif` dan pengujuian menggunakan Logistic Regression. Pengujian terhadap jumlah fitur 1 - 30 menggunakan 5-fold rossvalidation, dan diperoleh 26 fitur sbgai jumlah fitur optimal, dengan rata akurasi sekitar 98,07%.